In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

# https://stackoverflow.com/questions/21971449/how-do-i-increase-the-cell-width-of-the-jupyter-ipython-notebook-in-my-browser
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

from matplotlib import pyplot as plt
import numpy as np
from pathlib import Path
import json
import pickle
import random
from IPython.display import display, Image, clear_output
from PIL import Image as PILImage
# from pigeonXT import annotate
from collections import defaultdict
import sys
import os
from glob import glob
from pprint import pprint
import pandas as pd
import utils
import faiss
import networkx as nx
import textwrap
import ipyplot

In [ ]:
def show_imgs_sidebyside(imgs, captions, grayscale_imgs=None, title=None, figsize=None):
    """ Assume images already loaded """
    NUM_ROWS = 1
    IMGS_PER_ROW = len(imgs)
    f, ax = plt.subplots(NUM_ROWS, IMGS_PER_ROW, figsize=figsize if figsize else (30,4))
    
    for i, (img, caption) in enumerate(zip(imgs, captions)):
        ax[i].imshow(img, cmap='gray')
        # Add pad to increase the distance between the image and the title
        if isinstance(caption, str):
            ax[i].set_title(caption, pad=20)
        else:
            # If the caption is a dict; add pad manually to the title text:
            ax[i].set_title(caption.get('text', ''), pad=20, **{k: v for k, v in caption.items() if k != 'text'})
        ax[i].axes.xaxis.set_ticks([])
        ax[i].axes.yaxis.set_ticks([])
        if title is not None and i == 0:            
            ax[i].set_ylabel(**title, rotation=0, fontsize=14, labelpad=22)
    plt.tight_layout()
        
    if grayscale_imgs is not None:
        assert len(grayscale_imgs) == len(imgs)
        f1, ax1 = plt.subplots(NUM_ROWS, IMGS_PER_ROW, figsize=(16,6))
        for i, img in enumerate(grayscale_imgs):
            ax1[i].imshow(img)
            ax1[i].axes.xaxis.set_ticks([])
            ax1[i].axes.yaxis.set_ticks([])
            if title is not None and i == 0:
                ax1[i].set_ylabel(**title, rotation=0, fontsize=10, labelpad=22)
        plt.tight_layout()
    plt.show()


In [ ]:


def generate_individual_images_side_by_side(img_paths, captions, grayscale_imgs=None, output_dir="./ipynb_plot_output/", title=None, figsize=None):
    """
    Generates individual image files from a list of images and captions,
    conceptually arranged in a row.
    Args:
        img_paths (list): A list of paths.
        captions (list): A list of captions (strings or dictionaries for title properties).
        grayscale_imgs (list, optional): A list of grayscale image data. Defaults to None.
        base_filename (str, optional): The base filename for the generated images.
            Each image will be saved as f"{output_dir}/{base_filename}_{index}.png".
            Defaults to "image".
        output_dir (str, optional): The directory to save the images. Defaults to "./".
        title (dict, optional): A dictionary of title properties (applied to the conceptual "first" image). Defaults to None.
        figsize (tuple, optional): The figure size for each individual image. Defaults to None.
    """
    os.makedirs(output_dir, exist_ok=True)
    imgs = [np.array(PILImage.open(p)) for p in img_paths]
    output_paths = []

    for i, (img, caption) in enumerate(zip(imgs, captions)):
        fig, ax = plt.subplots(figsize=figsize)
        ax.imshow(img, cmap='gray')
        if isinstance(caption, str):
            wrapped_caption = textwrap.fill(caption, width=30)
            ax.set_title(wrapped_caption)
        else:
            # If the caption is a dict, assume the title text is under the key 'text'
            if 'text' in caption:
                caption['text'] = textwrap.fill(caption['text'], width=30)
            ax.set_title(**caption)
        ax.axis('off')  # Remove axes ticks and labels
        if title is not None and i == 0:
            ax.set_ylabel(**title, rotation=0, fontsize=8, labelpad=2)
        plt.tight_layout(pad=0)
        output_path = f"{output_dir}/{Path(img_paths[i]).with_suffix('').name}.png"
        plt.savefig(output_path, bbox_inches='tight', pad_inches=0)
        output_paths.append(output_path)
        plt.close(fig)
    ipyplot.plot_images(output_paths, max_images=len(output_paths), img_width=75, show_url=False, zoom_scale=3.0)



In [ ]:
def load_matching_run(exp_dir: str, exp_prefix: str):
    # glob for everything after prefix
    files = sorted(glob(f"{os.path.join(exp_dir, exp_prefix)}*.csv"))
    print(files)
    print(f'Found {len(files)} matching files. Characters: ')
    print(', '.join([fname[-5] for fname in files]))
    data = dict()
    for fname in files:
        queries = []
        candidates = []
        sims = []
        row_paths = []
        with open(fname) as f:
            for line in f.readlines():
                line = line.strip()
                this_query = line.split('|')[0]
                # count the number of '|' in the line
                num_sep = line.count('|')
                nc = num_sep // 2
                this_candidates = line.split('|')[1:nc + 1]
                this_sims = [float(s) for s in line.split('|')[nc+1:]]
                queries.append(this_query)
                candidates.append(this_candidates)
                sims.append(this_sims)
                row_paths.append([this_query] + this_candidates)
        data[fname[-5]] = {'row_paths': row_paths, 'queries': queries, 'candidates': candidates, 'sims': sims}
        # with open(fname, 'rb') as f:
        #     data[fname[-5]] = pickle.load(f)
    return data

In [ ]:
def get_image_titles(queries, candidates, sims):
    titles = []
    for i, p in enumerate([queries] + candidates):
        book_name = utils.get_book_name(Path(p).name)
        # text wrap book name
        printer = utils.get_printer_name(Path(p).name)
        book_name = textwrap.fill(book_name, width=30)
        page_num = utils.get_page_num(Path(p).name)
        line_num = utils.get_line_num(Path(p).name)
        char_loc = utils.get_char_loc(Path(p).name)
        if i == 0:  # query
            titles.append(f"Q{i}\n{printer}\n{book_name}\npage {page_num}, line {line_num}, loc {char_loc}" + "\n(Query)")
        else:  # candidate
            titles.append(f"M{i}\n{printer}\n{book_name}\npage {page_num}, line {line_num}, loc {char_loc}" + f"\n({sims[i-1]:.2f})")
    return titles
        


In [ ]:
data = load_matching_run('output/matching_results_may25/', 'faiss_top_50_results')


total_rows = sum([len(data[char]['queries']) for char in data.keys()])
print(f"Total rows: {total_rows}")
r = 0
max_r = 15
filter_char_class = False
filter_same_book_out = False
filter_same_page_out = True
min_sim = 0.3
topk = 10

for char in data.keys():
    # if char not in 'FGMN':
    #     continue
    print(f"Character {char} has {len(data[char]['queries'])} rows:")
    for i in range(len(data[char]['row_paths'])):
        if r > max_r:
            break
        # print(data[char]['row_paths'][i])
        query_char = utils.get_char(Path(data[char]['queries'][i]))
        query = data[char]['queries'][i]
        candidates = data[char]['candidates'][i]
        sims = data[char]['sims'][i]
        if filter_char_class:
            # only show candidates/sims that match the query character class
            valid_idxs = [n for n in range(len(candidates)) if utils.get_char(Path(candidates[n]).name) == query_char]
            candidates = [candidates[j] for j in valid_idxs]
            sims = [sims[j] for j in valid_idxs]
        if filter_same_book_out:
            valid_idxs = [n for n in range(len(candidates)) if utils.get_book_name(candidates[n]) != utils.get_book_name(query)]
            candidates = [candidates[j] for j in valid_idxs]
            sims = [sims[j] for j in valid_idxs]
        if filter_same_page_out:
            valid_idxs = [n for n in range(len(candidates)) if utils.get_page_num(candidates[n]) != utils.get_page_num(query)]
            candidates = [candidates[j] for j in valid_idxs]
            sims = [sims[j] for j in valid_idxs]
        if min_sim:
            valid_idxs = [n for n in range(len(candidates)) if sims[n] > min_sim]
            candidates = [candidates[j] for j in valid_idxs]
            sims = [sims[j] for j in valid_idxs]
        if len(candidates) == 0:
            print(f"No candidates for query {query} with character {query_char}. Skipping.")
            continue

        candidates = candidates[:topk]
        sims = sims[:topk]

        image_titles = get_image_titles(query, candidates, sims)
        show_imgs_sidebyside(
        # generate_individual_images_side_by_side(
            [np.array(PILImage.open(p)) for p in [query] + candidates],
            image_titles
            # [p for p in data[char]['row_paths'][i]],
            # [f"Q{i}\n{chr(10).join(Path(data[char]['queries'][i]).with_suffix('').name.split('_'))}",] 
            # + [
            #     f"M{j+1} ({data[char]['sims'][i][j]:0.2f})\n{chr(10).join(Path(data[char]['candidates'][i][j]).with_suffix('').name.split('_'))}"
            #     for j in range(10)
            # ]
        )
        r += 1